# Excel数据标签对比分析

## 功能说明
1. 从桌面读取两个Excel文件
2. 筛选稽核时间为 2026/4/30 的人工稽核结果
3. 对比稽核结果标签是否全部包含在数据分类分级表中

## 文件说明
- **表1**: cpms-资产稽核-字段清单-20260423.xlsx
- **表2**: 数据分类分级表-260311-v2(已自动还原).xlsx

In [25]:
import pandas as pd
import os

# ========================================
# 配置区域 - 如果需要修改列名，请在这里修改
# ========================================
time_column = '稽核时间'  # 表1中稽核时间的列名
audit_column = '人工稽核结果（数据标签）'  # ⚠️ 注意：使用中文括号！
classification_column_index = 6  # 表2中"数据标签"列的索引（G列=索引6）
target_date_str = '2026-04-30'  # 要筛选的目标日期
# ========================================

# 设置桌面路径（Windows系统）
desktop_path = os.path.expanduser("~/Desktop")

# 文件路径
file1 = os.path.join(desktop_path, 'cpms-资产稽核-字段清单-20260423.xlsx')
file2 = os.path.join(desktop_path, '数据分类分级表-260311-v2(已自动还原).xlsx')

print("="*60)
print("Excel数据标签对比分析工具")
print("="*60)
print(f"\n配置信息:")
print(f"  • 表1: {file1}")
print(f"  • 表2: {file2}")
print(f"  • 时间列: {time_column}")
print(f"  • 稽核结果列: {audit_column}")
print(f"  • 分类标签列索引: {classification_column_index} (G列)")
print(f"  • 目标日期: {target_date_str}")
print()

try:
    # 读取Excel文件
    print("正在读取Excel文件...")
    df1 = pd.read_excel(file1)
    df2 = pd.read_excel(file2)
    print(f"✓ 表1读取成功: {len(df1)} 行数据")
    print(f"✓ 表2读取成功: {len(df2)} 行数据\n")

    # 筛选目标日期的数据
    print("正在筛选数据...")
    df1[time_column] = pd.to_datetime(df1[time_column])
    target_date = pd.Timestamp(target_date_str)
    filtered_df = df1[df1[time_column].dt.date == target_date.date()]

    print(f"✓ 筛选出 {len(filtered_df)} 条 {target_date_str} 的记录\n")

    if len(filtered_df) == 0:
        print(f"️  警告：没有找到 {target_date_str} 的数据，请检查日期格式或列名")
    else:
        # 获取稽核结果标签集合（去重、清理空值）
        audit_labels = set(filtered_df[audit_column].dropna().astype(str).str.strip())
        audit_labels = {label for label in audit_labels if label and label != 'nan'}

        # 获取数据分类标签集合（使用列索引）
        classification_labels = set(df2.iloc[:, classification_column_index].dropna().astype(str).str.strip())
        classification_labels = {label for label in classification_labels if label and label != 'nan'}

        print(f"📊 统计信息:")
        print(f"  • 稽核结果唯一标签数: {len(audit_labels)} 个")
        print(f"  • 数据分类唯一标签数: {len(classification_labels)} 个\n")

        # 对比分析
        missing_labels = audit_labels - classification_labels
        common_labels = audit_labels & classification_labels
        only_in_classification = classification_labels - audit_labels

        # 输出结果
        print("="*60)
        print("对 比 分 析 报 告")
        print("="*60)

        # 判断是否完全包含
        if len(missing_labels) == 0:
            print(f"\n✅ 结论：所有稽核结果标签都包含在数据分类分级表中！")
            match_rate = len(common_labels) / len(audit_labels) * 100 if len(audit_labels) > 0 else 100
            print(f"   匹配率: {len(common_labels)}/{len(audit_labels)} = {match_rate:.2f}%")
        else:
            print(f"\n❌ 结论：有 {len(missing_labels)} 个稽核结果标签不在数据分类分级表中")
            match_rate = len(common_labels) / len(audit_labels) * 100 if len(audit_labels) > 0 else 0
            print(f"   匹配率: {len(common_labels)}/{len(audit_labels)} = {match_rate:.2f}%")

            print(f"\n 缺失的标签列表 ({len(missing_labels)} 个):")
            print("-"*60)
            for i, label in enumerate(sorted(missing_labels), 1):
                print(f"  {i:2d}. {label}")

        print(f"\n{'='*60}")

        # 可选：显示仅在数据分类中存在的标签

        print(f"\n{'='*60}")
        print("✅ 分析完成！")

except FileNotFoundError as e:
    print(f"\n❌ 错误：找不到文件")
    print(f"   详细信息: {e}")
except KeyError as e:
    print(f"\n❌ 错误：找不到指定的列")
    print(f"   详细信息: {e}")
    print(f"\n请检查配置区域的列名是否正确")
except Exception as e:
    print(f"\n❌ 发生错误: {type(e).__name__}")
    print(f"   详细信息: {e}")


Excel数据标签对比分析工具

配置信息:
  • 表1: C:\Users\15131/Desktop\cpms-资产稽核-字段清单-20260423.xlsx
  • 表2: C:\Users\15131/Desktop\数据分类分级表-260311-v2(已自动还原).xlsx
  • 时间列: 稽核时间
  • 稽核结果列: 人工稽核结果（数据标签）
  • 分类标签列索引: 6 (G列)
  • 目标日期: 2026-04-30

正在读取Excel文件...
✓ 表1读取成功: 17368 行数据
✓ 表2读取成功: 809 行数据

正在筛选数据...
✓ 筛选出 5195 条 2026-04-30 的记录

📊 统计信息:
  • 稽核结果唯一标签数: 81 个
  • 数据分类唯一标签数: 723 个

对 比 分 析 报 告

✅ 结论：所有稽核结果标签都包含在数据分类分级表中！
   匹配率: 81/81 = 100.00%


✅ 分析完成！


In [20]:
import pandas as pd
import os

# ========================================
# 配置区域 - 如果需要修改列名，请在这里修改
# ========================================
time_column = '稽核时间'  # 表1中稽核时间的列名
audit_column = '人工稽核结果（数据标签）'  # ⚠️ 注意：使用中文括号！
classification_column_index = 6  # 表2中"数据标签"列的索引（G列=索引6）
target_date_str = '2026-04-30'  # 要筛选的目标日期
# ========================================

# 设置桌面路径（Windows系统）
desktop_path = os.path.expanduser("~/Desktop")

# 文件路径
file1 = os.path.join(desktop_path, 'hbzqyydb-资产稽核-字段清单-20260310100755.xlsx')
file2 = os.path.join(desktop_path, '数据分类分级表-260311-v2(已自动还原).xlsx')

print("="*60)
print("Excel数据标签对比分析工具")
print("="*60)
print(f"\n配置信息:")
print(f"  • 表1: {file1}")
print(f"  • 表2: {file2}")
print(f"  • 时间列: {time_column}")
print(f"  • 稽核结果列: {audit_column}")
print(f"  • 分类标签列索引: {classification_column_index} (G列)")
print(f"  • 目标日期: {target_date_str}")
print()

try:
    # 读取Excel文件
    print("正在读取Excel文件...")
    df1 = pd.read_excel(file1)
    df2 = pd.read_excel(file2)
    print(f"✓ 表1读取成功: {len(df1)} 行数据")
    print(f"✓ 表2读取成功: {len(df2)} 行数据\n")

    # 筛选目标日期的数据
    print("正在筛选数据...")
    df1[time_column] = pd.to_datetime(df1[time_column])
    target_date = pd.Timestamp(target_date_str)
    filtered_df = df1[df1[time_column].dt.date == target_date.date()]

    print(f"✓ 筛选出 {len(filtered_df)} 条 {target_date_str} 的记录\n")

    if len(filtered_df) == 0:
        print(f"️  警告：没有找到 {target_date_str} 的数据，请检查日期格式或列名")
    else:
        # 获取稽核结果标签集合（去重、清理空值）
        audit_labels = set(filtered_df[audit_column].dropna().astype(str).str.strip())
        audit_labels = {label for label in audit_labels if label and label != 'nan'}

        # 获取数据分类标签集合（使用列索引）
        classification_labels = set(df2.iloc[:, classification_column_index].dropna().astype(str).str.strip())
        classification_labels = {label for label in classification_labels if label and label != 'nan'}

        print(f"📊 统计信息:")
        print(f"  • 稽核结果唯一标签数: {len(audit_labels)} 个")
        print(f"  • 数据分类唯一标签数: {len(classification_labels)} 个\n")

        # 对比分析
        missing_labels = audit_labels - classification_labels
        common_labels = audit_labels & classification_labels
        only_in_classification = classification_labels - audit_labels

        # 输出结果
        print("="*60)
        print("对 比 分 析 报 告")
        print("="*60)

        # 判断是否完全包含
        if len(missing_labels) == 0:
            print(f"\n✅ 结论：所有稽核结果标签都包含在数据分类分级表中！")
            match_rate = len(common_labels) / len(audit_labels) * 100 if len(audit_labels) > 0 else 100
            print(f"   匹配率: {len(common_labels)}/{len(audit_labels)} = {match_rate:.2f}%")
        else:
            print(f"\n❌ 结论：有 {len(missing_labels)} 个稽核结果标签不在数据分类分级表中")
            match_rate = len(common_labels) / len(audit_labels) * 100 if len(audit_labels) > 0 else 0
            print(f"   匹配率: {len(common_labels)}/{len(audit_labels)} = {match_rate:.2f}%")

            print(f"\n 缺失的标签列表 ({len(missing_labels)} 个):")
            print("-"*60)
            for i, label in enumerate(sorted(missing_labels), 1):
                print(f"  {i:2d}. {label}")

        print(f"\n{'='*60}")

        # 可选：显示仅在数据分类中存在的标签

        print(f"\n{'='*60}")
        print("✅ 分析完成！")

except FileNotFoundError as e:
    print(f"\n❌ 错误：找不到文件")
    print(f"   详细信息: {e}")
except KeyError as e:
    print(f"\n❌ 错误：找不到指定的列")
    print(f"   详细信息: {e}")
    print(f"\n请检查配置区域的列名是否正确")
except Exception as e:
    print(f"\n❌ 发生错误: {type(e).__name__}")
    print(f"   详细信息: {e}")


Excel数据标签对比分析工具

配置信息:
  • 表1: C:\Users\15131/Desktop\hbzqyydb-资产稽核-字段清单-20260310100755.xlsx
  • 表2: C:\Users\15131/Desktop\数据分类分级表-260311-v2(已自动还原).xlsx
  • 时间列: 稽核时间
  • 稽核结果列: 人工稽核结果（数据标签）
  • 分类标签列索引: 6 (G列)
  • 目标日期: 2026-04-30

正在读取Excel文件...
✓ 表1读取成功: 4367 行数据
✓ 表2读取成功: 809 行数据

正在筛选数据...
✓ 筛选出 1028 条 2026-04-30 的记录

📊 统计信息:
  • 稽核结果唯一标签数: 31 个
  • 数据分类唯一标签数: 723 个

对 比 分 析 报 告

❌ 结论：有 2 个稽核结果标签不在数据分类分级表中
   匹配率: 29/31 = 93.55%

 缺失的标签列表 (2 个):
------------------------------------------------------------
   1. 城市编码
   2. 市场营销数据及分析报告


📝 数据分类中额外存在的标签 (694 个):
------------------------------------------------------------
   1. AC（接入点）
   2. AKEY
   3. APN
   4. APP使用日志
   5. APP偏好
   6. AP（接入控制器）
   7. CI
   8. CP/SP业务订购数据
   9. CP/SP基本资料
  10. CP/SP结算数据
  11. CP/SP识别信息
  12. Cookie内容
  13. DDF（数字配线架）
  14. DDM（数字诊断监视功能模块）
  15. ESN
  16. ICCID
  17. IDC/ISP告警信息
  18. IMEI
  19. IMSI
  20. IMS系统信息
  ... 还有 674 个标签

✅ 分析完成！


In [23]:
import pandas as pd
import os

# ========================================
# 配置区域 - 如果需要修改列名，请在这里修改
# ========================================
time_column = '稽核时间'  # 表1中稽核时间的列名
audit_column = '人工稽核结果（数据标签）'  # ⚠️ 注意：使用中文括号！
classification_column_index = 6  # 表2中"数据标签"列的索引（G列=索引6）
target_date_str = '2026-04-30'  # 要筛选的目标日期
# ========================================

# 设置桌面路径（Windows系统）
desktop_path = os.path.expanduser("~/Desktop")

# 文件路径
file1 = os.path.join(desktop_path, 'hbzqyydb-资产稽核-字段清单-20260310100755.xlsx')
file2 = os.path.join(desktop_path, '数据分类分级表-260311-v2(已自动还原).xlsx')

print("="*60)
print("Excel数据标签对比分析工具")
print("="*60)
print(f"\n配置信息:")
print(f"  • 表1: {file1}")
print(f"  • 表2: {file2}")
print(f"  • 时间列: {time_column}")
print(f"  • 稽核结果列: {audit_column}")
print(f"  • 分类标签列索引: {classification_column_index} (G列)")
print(f"  • 目标日期: {target_date_str}")
print()

try:
    # 读取Excel文件
    print("正在读取Excel文件...")
    df1 = pd.read_excel(file1)
    df2 = pd.read_excel(file2)
    print(f"✓ 表1读取成功: {len(df1)} 行数据")
    print(f"✓ 表2读取成功: {len(df2)} 行数据\n")

    # 筛选目标日期的数据
    print("正在筛选数据...")
    df1[time_column] = pd.to_datetime(df1[time_column])
    target_date = pd.Timestamp(target_date_str)
    filtered_df = df1[df1[time_column].dt.date == target_date.date()]

    print(f"✓ 筛选出 {len(filtered_df)} 条 {target_date_str} 的记录\n")

    if len(filtered_df) == 0:
        print(f"️  警告：没有找到 {target_date_str} 的数据，请检查日期格式或列名")
    else:
        # 获取稽核结果标签集合（去重、清理空值）
        audit_labels = set(filtered_df[audit_column].dropna().astype(str).str.strip())
        audit_labels = {label for label in audit_labels if label and label != 'nan'}

        # 获取数据分类标签集合（使用列索引）
        classification_labels = set(df2.iloc[:, classification_column_index].dropna().astype(str).str.strip())
        classification_labels = {label for label in classification_labels if label and label != 'nan'}

        print(f"📊 统计信息:")
        print(f"  • 稽核结果唯一标签数: {len(audit_labels)} 个")
        print(f"  • 数据分类唯一标签数: {len(classification_labels)} 个\n")

        # 对比分析
        missing_labels = audit_labels - classification_labels
        common_labels = audit_labels & classification_labels
        only_in_classification = classification_labels - audit_labels

        # 输出结果
        print("="*60)
        print("对 比 分 析 报 告")
        print("="*60)

        # 判断是否完全包含
        if len(missing_labels) == 0:
            print(f"\n✅ 结论：所有稽核结果标签都包含在数据分类分级表中！")
            match_rate = len(common_labels) / len(audit_labels) * 100 if len(audit_labels) > 0 else 100
            print(f"   匹配率: {len(common_labels)}/{len(audit_labels)} = {match_rate:.2f}%")
        else:
            print(f"\n❌ 结论：有 {len(missing_labels)} 个稽核结果标签不在数据分类分级表中")
            match_rate = len(common_labels) / len(audit_labels) * 100 if len(audit_labels) > 0 else 0
            print(f"   匹配率: {len(common_labels)}/{len(audit_labels)} = {match_rate:.2f}%")

            print(f"\n 缺失的标签列表 ({len(missing_labels)} 个):")
            print("-"*60)
            for i, label in enumerate(sorted(missing_labels), 1):
                print(f"  {i:2d}. {label}")

        print(f"\n{'='*60}")

        # 可选：显示仅在数据分类中存在的标签

        print(f"\n{'='*60}")
        print("✅ 分析完成！")

except FileNotFoundError as e:
    print(f"\n❌ 错误：找不到文件")
    print(f"   详细信息: {e}")
except KeyError as e:
    print(f"\n❌ 错误：找不到指定的列")
    print(f"   详细信息: {e}")
    print(f"\n请检查配置区域的列名是否正确")
except Exception as e:
    print(f"\n❌ 发生错误: {type(e).__name__}")
    print(f"   详细信息: {e}")


Excel数据标签对比分析工具

配置信息:
  • 表1: C:\Users\15131/Desktop\hbzqyydb-资产稽核-字段清单-20260310100755.xlsx
  • 表2: C:\Users\15131/Desktop\数据分类分级表-260311-v2(已自动还原).xlsx
  • 时间列: 稽核时间
  • 稽核结果列: 人工稽核结果（数据标签）
  • 分类标签列索引: 6 (G列)
  • 目标日期: 2026-04-30

正在读取Excel文件...
✓ 表1读取成功: 4367 行数据
✓ 表2读取成功: 809 行数据

正在筛选数据...
✓ 筛选出 1028 条 2026-04-30 的记录

📊 统计信息:
  • 稽核结果唯一标签数: 30 个
  • 数据分类唯一标签数: 723 个

对 比 分 析 报 告

✅ 结论：所有稽核结果标签都包含在数据分类分级表中！
   匹配率: 30/30 = 100.00%


📝 数据分类中额外存在的标签 (693 个):
------------------------------------------------------------
   1. AC（接入点）
   2. AKEY
   3. APN
   4. APP使用日志
   5. APP偏好
   6. AP（接入控制器）
   7. CI
   8. CP/SP业务订购数据
   9. CP/SP基本资料
  10. CP/SP结算数据
  11. CP/SP识别信息
  12. Cookie内容
  13. DDF（数字配线架）
  14. DDM（数字诊断监视功能模块）
  15. ESN
  16. ICCID
  17. IDC/ISP告警信息
  18. IMEI
  19. IMSI
  20. IMS系统信息
  ... 还有 673 个标签

✅ 分析完成！


In [21]:
import pandas as pd
import os

# ========================================
# 配置区域 - 如果需要修改列名，请在这里修改
# ========================================
time_column = '稽核时间'  # 表1中稽核时间的列名
audit_column = '人工稽核结果（数据标签）'  # ⚠️ 注意：使用中文括号！
classification_column_index = 6  # 表2中"数据标签"列的索引（G列=索引6）
target_date_str = '2026-04-30'  # 要筛选的目标日期
# ========================================

# 设置桌面路径（Windows系统）
desktop_path = os.path.expanduser("~/Desktop")

# 文件路径
file1 = os.path.join(desktop_path, 'cpms-资产稽核-字段清单-20260423.xlsx')
file2 = os.path.join(desktop_path, '数据分类分级表-260311-v2(已自动还原).xlsx')

print("="*60)
print("Excel数据标签对比分析工具")
print("="*60)
print(f"\n配置信息:")
print(f"  • 表1: {file1}")
print(f"  • 表2: {file2}")
print(f"  • 时间列: {time_column}")
print(f"  • 稽核结果列: {audit_column}")
print(f"  • 分类标签列索引: {classification_column_index} (G列)")
print(f"  • 目标日期: {target_date_str}")
print()

try:
    # 读取Excel文件
    print("正在读取Excel文件...")
    df1 = pd.read_excel(file1)
    df2 = pd.read_excel(file2)
    print(f"✓ 表1读取成功: {len(df1)} 行数据")
    print(f"✓ 表2读取成功: {len(df2)} 行数据\n")

    # 筛选目标日期的数据
    print("正在筛选数据...")
    df1[time_column] = pd.to_datetime(df1[time_column])
    target_date = pd.Timestamp(target_date_str)
    filtered_df = df1[df1[time_column].dt.date == target_date.date()]

    print(f"✓ 筛选出 {len(filtered_df)} 条 {target_date_str} 的记录\n")

    if len(filtered_df) == 0:
        print(f"️  警告：没有找到 {target_date_str} 的数据，请检查日期格式或列名")
    else:
        # 获取稽核结果标签集合（去重、清理空值）
        audit_labels = set(filtered_df[audit_column].dropna().astype(str).str.strip())
        audit_labels = {label for label in audit_labels if label and label != 'nan'}

        # 获取数据分类标签集合（使用列索引）
        classification_labels = set(df2.iloc[:, classification_column_index].dropna().astype(str).str.strip())
        classification_labels = {label for label in classification_labels if label and label != 'nan'}

        print(f"📊 统计信息:")
        print(f"  • 稽核结果唯一标签数: {len(audit_labels)} 个")
        print(f"  • 数据分类唯一标签数: {len(classification_labels)} 个\n")

        # 对比分析
        missing_labels = audit_labels - classification_labels
        common_labels = audit_labels & classification_labels
        only_in_classification = classification_labels - audit_labels

        # 输出结果
        print("="*60)
        print("对 比 分 析 报 告")
        print("="*60)

        # 判断是否完全包含
        if len(missing_labels) == 0:
            print(f"\n✅ 结论：所有稽核结果标签都包含在数据分类分级表中！")
            match_rate = len(common_labels) / len(audit_labels) * 100 if len(audit_labels) > 0 else 100
            print(f"   匹配率: {len(common_labels)}/{len(audit_labels)} = {match_rate:.2f}%")
        else:
            print(f"\n❌ 结论：有 {len(missing_labels)} 个稽核结果标签不在数据分类分级表中")
            match_rate = len(common_labels) / len(audit_labels) * 100 if len(audit_labels) > 0 else 0
            print(f"   匹配率: {len(common_labels)}/{len(audit_labels)} = {match_rate:.2f}%")

            print(f"\n 缺失的标签列表 ({len(missing_labels)} 个):")
            print("-"*60)
            for i, label in enumerate(sorted(missing_labels), 1):
                print(f"  {i:2d}. {label}")

        print(f"\n{'='*60}")


        print(f"\n{'='*60}")
        print("✅ 分析完成！")

except FileNotFoundError as e:
    print(f"\n❌ 错误：找不到文件")
    print(f"   详细信息: {e}")
except KeyError as e:
    print(f"\n❌ 错误：找不到指定的列")
    print(f"   详细信息: {e}")
    print(f"\n请检查配置区域的列名是否正确")
except Exception as e:
    print(f"\n❌ 发生错误: {type(e).__name__}")
    print(f"   详细信息: {e}")


Excel数据标签对比分析工具

配置信息:
  • 表1: C:\Users\15131/Desktop\cpms-资产稽核-字段清单-20260423.xlsx
  • 表2: C:\Users\15131/Desktop\数据分类分级表-260311-v2(已自动还原).xlsx
  • 时间列: 稽核时间
  • 稽核结果列: 人工稽核结果（数据标签）
  • 分类标签列索引: 6 (G列)
  • 目标日期: 2026-04-30

正在读取Excel文件...
✓ 表1读取成功: 17368 行数据
✓ 表2读取成功: 809 行数据

正在筛选数据...
✓ 筛选出 5195 条 2026-04-30 的记录

📊 统计信息:
  • 稽核结果唯一标签数: 81 个
  • 数据分类唯一标签数: 723 个

对 比 分 析 报 告

✅ 结论：所有稽核结果标签都包含在数据分类分级表中！
   匹配率: 81/81 = 100.00%


📝 数据分类中额外存在的标签 (642 个):
------------------------------------------------------------
   1. AC（接入点）
   2. AKEY
   3. APN
   4. APP使用日志
   5. APP偏好
   6. AP（接入控制器）
   7. CI
   8. CP/SP业务订购数据
   9. CP/SP基本资料
  10. CP/SP结算数据
  11. CP/SP识别信息
  12. Cookie内容
  13. DDF（数字配线架）
  14. DDM（数字诊断监视功能模块）
  15. ESN
  16. ICCID
  17. IDC/ISP告警信息
  18. IMEI
  19. IMSI
  20. IMS系统信息
  ... 还有 622 个标签

✅ 分析完成！
